Functions in the re module


Below is a list of the most common functions in the re module
```
re.match(pattern, str)
re.search(pattern, str)
re.findall(pattern, str)
re.finditer(pattern, str)
re.sub(pattern, replacement, str, count=0)```

Functions ```match``` and ```search``` return a match object. A match object describes the found occurence.

The function ```findall``` returns a list of all the occurences of the pattern. The elements in the list are strings.

The function ```finditer``` works like ```findall``` function except that instead of returning a list, it returns an iterator whose items are match objects.

The function sub replaces all the occurences of the pattern in str with the string replacement and returns the new string.

## The Five Main `re` Functions — Simply, With Examples

Three of these are old friends from our discussions; two are new. Let's use one shared example string throughout so you can compare them directly:

```python
import re

s = "cat 42 dog 7 bird 100"
```

---

### 1. `re.match(pattern, s)` — Try at the START Only

Attempts the match **only at position 0** (the function anchors, as you correctly dissected earlier!). Returns a **match object** or `None`.

```python
re.match(r'\w+', s)          # → <re.Match; span=(0, 3), match='cat'>  ✓
re.match(r'\d+', s)          # → None  — position 0 is 'c', not a digit
```

**Use for:** validating that input *begins with* (or has) a required shape — dates, IDs, tokens.

---

### 2. `re.search(pattern, s)` — Find the FIRST Match Anywhere

Scans position by position until the first hit. Returns a **match object** or `None`.

```python
m = re.search(r'\d+', s)
m.group()                    # → '42'      the first number, wherever it is
m.span()                     # → (4, 6)
```

**Use for:** *"does this appear, and where?"* — one answer is enough.

---

### 3. `re.findall(pattern, s)` — ALL Matches, as a List of Strings

The whole harvest at once — plain strings, no match objects:

```python
re.findall(r'\d+', s)        # → ['42', '7', '100']
```

And remember the group twist you asked about: with parentheses in the pattern, you get **only the captured parts**:

```python
re.findall(r'(\w+) \d+', s)  # → ['cat', 'dog', 'bird']
```

**Use for:** *"give me every occurrence"* — then process the list with your usual tools (`len`, comprehensions, `map(int, ...)`).

---

### 4. `re.finditer(pattern, s)` — All Matches, as a LAZY Stream of Match Objects 🆕

Same coverage as `findall`, but two differences: it yields **match objects** (with positions!), and it's a **lazy iterator** — your generator knowledge applies directly:

```python
it = re.finditer(r'\d+', s)
print(it)                    # → <callable_iterator ...>   nothing computed yet!

for m in it:
    print(m.group(), "at", m.span())
```

```
42 at (4, 6)
7 at (11, 12)
100 at (18, 21)
```

**`findall` vs `finditer`:**

| | `findall` | `finditer` |
|---|---|---|
| Returns | list of strings | lazy stream of match objects |
| Positions available? | ✗ | ✓ `.span()`, `.start()` |
| Memory on huge text | whole list at once | one match at a time |
| Reusable? | ✓ it's a list | ✗ one pass (like all iterators) |

**Use for:** when you need **where** each match is, or the text is huge.

---

### 5. `re.sub(pattern, replacement, s, count=0)` — Find and REPLACE 🆕

The only one that **produces new text** (strings are immutable — it returns a modified copy, original untouched):

```python
re.sub(r'\d+', '#', s)           # → 'cat # dog # bird #'
re.sub(r'\d+', '#', s, count=2)  # → 'cat # dog # bird 100'   only first 2
```

`count=0` (the default) means *replace all* — a slightly odd convention, but that's what it is.

You've met `sub` twice already! Removing punctuation (`re.sub(r'[^\w\s]', '', s)` — replace with nothing = delete) and whole-word replacement (`re.sub(r'\bis\b', 'was', ...)`).

Bonus: the replacement can reuse captures via backreferences — here swapping word–number pairs:

```python
re.sub(r'(\w+) (\d+)', r'\2 \1', s)
# → '42 cat 7 dog 100 bird'
```

**Use for:** cleaning, censoring, reformatting text.

---

### The Decision Table

| You want... | Use |
|---|---|
| "Does it start with / conform to X?" | `match` |
| "First occurrence + its position" | `search` |
| "All occurrences as simple strings" | `findall` |
| "All occurrences with positions / big text" | `finditer` |
| "Replace occurrences" | `sub` |

**Return-type map** (the part that trips people up):

```
match, search  → match object or None      (check with if m:)
findall        → list of strings           (may be empty [])
finditer       → lazy iterator of match objects
sub            → a new string
```

One family, five verbs: **check the start, find one, collect all, stream all, rewrite.** 🎯

An example: The following program will replace all "she" words with "he"

In [4]:
import re

str  = "She goes where she wants to, she's sheriff"

#re.sub(r'\b[Ss]he\b', 'he', str)
# '\b[Ss]he\b'
# \b \b = Matching whole word only
#[Ss]he = Start with S or s,  then he  = She or she
# She(he) goes where she(he) wants to, she(he)'s sheriff  = he goes where he wants to, he's sheriff
newstr = re.sub(r'\b[Ss]he\b', 'he', str)
print(newstr)



he goes where he wants to, he's sheriff


The sub function can also use backreferences to refer to the matched string.

The backreferences ```\1, \2```, and so on, refer to the groups of the pattern, in order. An example:

In [8]:
str = """He is a timelord.
He has a Tardis."""
newstr = re.sub(r'(\b[Hh]e\b)', r'\1 (The Doctor)', str, count=1)
print(newstr)


He (The Doctor) is a timelord.
He has a Tardis.


## Two Things Happening Here — The Output ✓ and a Warning ⚠

The code **works** — you got the right output. But Python is nudging you about style. Let's decode both.

---

### First — Why the Output Is Right

```python
re.sub(r'(\b[Hh]e\b)', r'\1 (The Doctor)', str, 1)
```

Every piece is from your recent lessons:

| Piece | You know this as... | Job here |
|---|---|---|
| `\b...\b` | word boundaries | whole word `He`/`he` only — not the `he` inside "T**he**" or "ot**he**r"! |
| `[Hh]e` | bracket set | `He` or `he` |
| `( ... )` | capture group | remember which one was found |
| `\1` in replacement | backreference | paste the captured word back |
| `1` (last argument) | the `count` parameter | replace only the FIRST occurrence |

So the replacement `r'\1 (The Doctor)'` means: *"put back exactly what was matched, then append ` (The Doctor)`."* The capture is what preserves the original capitalization — `He` stays `He`. And `count=1` is why the second line's `He` was left alone:

```
He (The Doctor) is a timelord.    ← first He: annotated
He has a Tardis.                  ← second He: untouched (count exhausted)
```

Nice pattern — genuinely well constructed! 🎯

---

### Second — What the Warning Is About

```
DeprecationWarning: 'count' is passed as positional argument
```

**Deprecation** = *"this still works today, but it's marked for removal — please stop using it."* It's Python's polite advance notice, not an error.

The issue: you passed `1` **positionally** (by its place in the argument list):

```python
re.sub(pattern, replacement, str, 1)     # ⚠ what does bare '1' mean here?
```

Python wants it **as a keyword** (recall your keyword-arguments material — `sep=`, `end=`, `degree=`...):

```python
re.sub(pattern, replacement, str, count=1)    # ✓ self-documenting, no warning
```

**Why do they demand this?** Readability and safety. `re.sub` also has a `flags` parameter after `count` — with bare positional numbers, `re.sub(p, r, s, 1)` vs a mistaken `re.sub(p, r, s, re.IGNORECASE)` land in the wrong slots silently. `count=1` is unambiguous to both Python and the human reader. Newer Python versions (3.13+) enforce keyword-only for these parameters — hence the advance warning now.

---

### Third — A Bug You Didn't Notice: `str` Shadowing!

Your variable name:

```python
str = """He is a timelord. ..."""
```

This **rebinds the name `str`** — the builtin type constructor — to your text. Third appearance of the shadowing lesson (after `random.py` and your `hypotenuse` recursion)! Later in the same session:

```python
str(42)      # ✗ TypeError: 'str' object is not callable
#              — 'str' is now your Doctor Who text, not the function!
```

Rename to `text` or `s`.

---

### The Cleaned-Up Version

```python
import re

text = """He is a timelord.
He has a Tardis."""

newtext = re.sub(r'(\b[Hh]e\b)', r'\1 (The Doctor)', text, count=1)
print(newtext)
```

```
He (The Doctor) is a timelord.
He has a Tardis.
```

Same output, no warning, no shadowed builtin.

> **Summary:** the regex logic was already correct — the warning only asked you to write `count=1` explicitly instead of a bare positional `1`, and the silent hazard was naming your string `str`. Three small polish items, zero logic changes. 🎯